# MLIP relaxation sweep (Co–Bi demo)

This notebook relaxes a batch of candidate crystal structures with CHGNet (or MACE-MP) and summarises the results.

**What it does:**
1. Loads candidate CIFs from `data/candidates/Co-Bi/` (or generates demo structures if none exist).
2. Relaxes each structure with the chosen MLIP.
3. Saves relaxed CIFs and a results CSV.
4. Plots energy-per-atom, forces, and volume across candidates.

MLIP energies are *screening* predictions — they prioritise candidates for DFT, not prove stability.

```bash
pip install -e ".[notebook]"
jupyter lab
```

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

REPO = Path.cwd().resolve()
if not (REPO / "data").is_dir() and (REPO.parent / "data").is_dir():
    REPO = REPO.parent
if not (REPO / "data").is_dir():
    raise FileNotFoundError(
        f"Expected data/ under {REPO}. Open the notebook from the hullgap repo root."
    )

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

print("REPO =", REPO)

## 1. Configuration

Edit these to point at your candidate directory and adjust relaxation settings.

In [ ]:
SYSTEM = "Co-Bi"
MODEL = "chgnet"            # "chgnet" or "mace"
FMAX = 0.05                  # eV/Å  (screening default)
MAX_STEPS = 300              # optimiser cap
RELAX_CELL = True

CANDIDATE_DIR = REPO / f"data/candidates/{SYSTEM}"
RELAXED_DIR = REPO / f"data/relaxed/{SYSTEM}"
RESULTS_DIR = REPO / "data/results"

RELAXED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load (or generate) candidate CIFs

If `data/candidates/Co-Bi/` has CIF files the pipeline produced, those are used.
Otherwise we generate a handful of demo structures by prototype substitution so the notebook is self-contained.

In [ ]:
from pymatgen.core import Lattice, Structure


def _make_demo_candidates(out_dir: Path) -> list[Path]:
    """Generate a few synthetic Co-Bi structures for testing."""
    out_dir.mkdir(parents=True, exist_ok=True)
    demos = [
        # CsCl-type CoBi
        ("demo_cscl_CoBi", Structure(
            Lattice.cubic(3.15),
            ["Co", "Bi"],
            [[0, 0, 0], [0.5, 0.5, 0.5]],
        )),
        # NaCl-type CoBi
        ("demo_nacl_CoBi", Structure(
            Lattice.cubic(5.0),
            ["Co", "Co", "Co", "Co", "Bi", "Bi", "Bi", "Bi"],
            [
                [0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5],
                [0.5, 0, 0], [0, 0.5, 0], [0, 0, 0.5], [0.5, 0.5, 0.5],
            ],
        )),
        # ZnS-type CoBi
        ("demo_zns_CoBi", Structure(
            Lattice.cubic(5.3),
            ["Co", "Co", "Co", "Co", "Bi", "Bi", "Bi", "Bi"],
            [
                [0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5],
                [0.25, 0.25, 0.25], [0.75, 0.75, 0.25],
                [0.75, 0.25, 0.75], [0.25, 0.75, 0.75],
            ],
        )),
        # Hexagonal Co3Bi
        ("demo_hex_Co3Bi", Structure(
            Lattice.hexagonal(5.0, 4.0),
            ["Co", "Co", "Co", "Bi"],
            [
                [0.333, 0.0, 0.0], [0.0, 0.333, 0.0], [0.667, 0.667, 0.0],
                [0.0, 0.0, 0.5],
            ],
        )),
        # Tetragonal CoBi2
        ("demo_tet_CoBi2", Structure(
            Lattice.tetragonal(4.5, 6.0),
            ["Co", "Bi", "Bi"],
            [[0, 0, 0], [0.5, 0.5, 0.25], [0.5, 0.5, 0.75]],
        )),
    ]
    paths = []
    for name, struct in demos:
        p = out_dir / f"{name}.cif"
        struct.to(filename=str(p))
        paths.append(p)
    return paths


cif_files = sorted(CANDIDATE_DIR.glob("*.cif")) if CANDIDATE_DIR.is_dir() else []

if not cif_files:
    print(f"No CIFs in {CANDIDATE_DIR} — generating demo candidates.")
    cif_files = _make_demo_candidates(CANDIDATE_DIR)

print(f"\n{len(cif_files)} candidate(s):")
for p in cif_files:
    s = Structure.from_file(p)
    print(f"  {p.stem:30s}  {s.composition.reduced_formula:8s}  {len(s)} atoms")

## 3. Load MLIP calculator

In [ ]:
from hullgap.calculators import get_calculator

calc = get_calculator(MODEL)
print(f"Calculator ready: {MODEL}")

## 4. Relax all candidates

In [ ]:
from hullgap.relax import relax_structure

results = []
for cif_path in tqdm(cif_files, desc=f"Relaxing ({MODEL})"):
    out_cif = RELAXED_DIR / cif_path.name
    rec = relax_structure(
        input_file=str(cif_path),
        output_file=str(out_cif),
        model=MODEL,
        fmax=FMAX,
        max_steps=MAX_STEPS,
        relax_cell=RELAX_CELL,
        _calculator=calc,
    )
    results.append(rec)

df = pd.DataFrame(results)

csv_path = RESULTS_DIR / f"relaxation_results_{SYSTEM}_{MODEL}.csv"
df.to_csv(csv_path, index=False)
print(f"\nResults saved to {csv_path}")
print(f"Converged: {(df['status'] == 'converged').sum()}/{len(df)}")

## 5. Results table

In [ ]:
display_cols = [
    "candidate_id", "formula", "status",
    "energy_per_atom_eV", "max_force_eV_A",
    "volume_per_atom", "n_steps",
]
df_display = df[display_cols].copy()
df_display = df_display.sort_values("energy_per_atom_eV")
df_display.style.format({
    "energy_per_atom_eV": "{:.4f}",
    "max_force_eV_A": "{:.2e}",
    "volume_per_atom": "{:.2f}",
}).background_gradient(subset=["energy_per_atom_eV"], cmap="RdYlGn_r")

## 6. Plots

In [ ]:
ok = df[df["status"] != "failed_relaxation"].copy()
ok = ok.sort_values("energy_per_atom_eV")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Energy per atom
ax = axes[0]
colors = ["#2ecc71" if s == "converged" else "#e67e22" for s in ok["status"]]
ax.barh(ok["candidate_id"], ok["energy_per_atom_eV"], color=colors)
ax.set_xlabel("Energy / atom (eV)")
ax.set_title("MLIP energy per atom")
ax.invert_yaxis()

# Residual max force
ax = axes[1]
ax.barh(ok["candidate_id"], ok["max_force_eV_A"], color="#3498db")
ax.axvline(FMAX, color="red", ls="--", lw=1, label=f"fmax={FMAX}")
ax.set_xlabel("Max force (eV/\u00c5)")
ax.set_title("Residual forces")
ax.set_xscale("log")
ax.legend()
ax.invert_yaxis()

# Volume per atom
ax = axes[2]
ax.barh(ok["candidate_id"], ok["volume_per_atom"], color="#9b59b6")
ax.set_xlabel("Volume / atom (\u00c5\u00b3)")
ax.set_title("Relaxed volume")
ax.invert_yaxis()

fig.suptitle(f"{SYSTEM} MLIP relaxation sweep ({MODEL})", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(REPO / f"reports/figures/{SYSTEM}_relaxation_sweep_{MODEL}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to reports/figures/{SYSTEM}_relaxation_sweep_{MODEL}.png")

## 7. Energy ranking

The lowest energy-per-atom candidates are the most promising for hull scoring.
Remember: these are MLIP screening energies, not DFT ground truth.

In [ ]:
ranking = ok[["candidate_id", "formula", "energy_per_atom_eV", "volume_per_atom", "n_steps"]].copy()
ranking["rank"] = range(1, len(ranking) + 1)
ranking = ranking[["rank"] + [c for c in ranking.columns if c != "rank"]]
print(f"Top {len(ranking)} candidates by MLIP energy ({MODEL}):")
ranking

## 8. (Optional) Compare with a second model

Uncomment and re-run to relax the same candidates with MACE-MP for cross-validation.

```python
MODEL2 = "mace"  # requires: pip install -e ".[mace]"
calc2 = get_calculator(MODEL2)
results2 = []
for cif_path in tqdm(cif_files, desc=f"Relaxing ({MODEL2})"):
    out_cif = RELAXED_DIR / f"{cif_path.stem}_{MODEL2}.cif"
    rec = relax_structure(
        input_file=str(cif_path),
        output_file=str(out_cif),
        model=MODEL2, fmax=FMAX, max_steps=MAX_STEPS,
        relax_cell=RELAX_CELL, _calculator=calc2,
    )
    results2.append(rec)
df2 = pd.DataFrame(results2)
df2.to_csv(RESULTS_DIR / f"relaxation_results_{SYSTEM}_{MODEL2}.csv", index=False)

merged = ok[["candidate_id", "energy_per_atom_eV"]].merge(
    df2[["candidate_id", "energy_per_atom_eV"]],
    on="candidate_id", suffixes=(f"_{MODEL}", f"_{MODEL2}"),
)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(merged[f"energy_per_atom_eV_{MODEL}"], merged[f"energy_per_atom_eV_{MODEL2}"], s=60)
lo = min(merged.filter(like="energy").min())
hi = max(merged.filter(like="energy").max())
ax.plot([lo, hi], [lo, hi], "k--", lw=1)
ax.set_xlabel(f"{MODEL} energy/atom (eV)")
ax.set_ylabel(f"{MODEL2} energy/atom (eV)")
ax.set_title("Cross-model energy comparison")
plt.tight_layout()
plt.show()
```